# Diagnóstico 03 — Alocações nativas e Memory Pool do PyArrow

## Objetivo

Investigar o aumento de USS observado durante a leitura das bases `grupo` e `dsp`, antes da conversão para pandas.

O notebook mede simultaneamente:

- RSS e USS do processo;
- bytes atualmente alocados no Memory Pool do PyArrow;
- pico de alocação registrado pelo Memory Pool;
- `pa.total_allocated_bytes()`;
- tamanho dos buffers da tabela Arrow;
- tamanho do arquivo e metadados Parquet;
- efeito de `memory_map=True` e `memory_map=False`;
- comportamento após liberar a tabela e fechar o arquivo.

Os cenários são executados em processos filhos independentes para evitar que uma leitura anterior contamine a seguinte.

**Não altera o dashboard nem os arquivos Parquet.** O foco inicial é `grupo` e `dsp`.

## 1. Configuração

Execute a partir da raiz de `app_render`. A configuração e as projeções reproduzem os caminhos e colunas do `features_names.py`.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]
    PROJECT_ROOT = next(
        (p.resolve() for p in candidatos if (p / "dash_utils" / "config.py").exists()),
        None
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei dash_utils/config.py. Ajuste PROJECT_ROOT.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
CONSUMO_DIR = Path(config.CONSUMO_DIR)
UF = "PA"
REFERENCIA = "ESC"   # "ESC", "PROVA" ou None

UF_DIR = CONSUMO_DIR / f"uf={UF}"
ARQUIVOS = {
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

COLUNAS = {
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
}

print("Projeto:", PROJECT_ROOT)
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Memory pool padrão:", pa.default_memory_pool().backend_name)
print("UF:", UF, "| referência:", REFERENCIA)

for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    print(f"{nome}: {caminho.stat().st_size / 1024**2:.2f} MB | {caminho}")

Projeto: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Python: 3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]
Pandas: 3.1.0
PyArrow: 26.0.1
Memory pool padrão: mimalloc
UF: PA | referência: ESC
grupo: 8.57 MB | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_grupos_municipal.parquet
dsp: 10.10 MB | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_sexo_faixa_etaria_municipal.parquet


## 2. Referência de metadados e estrutura Parquet

Esta etapa não carrega as colunas de dados. Registra row groups, tamanho comprimido/descomprimido e encodings para contextualizar as medições seguintes.

In [2]:
linhas_meta=[]
for base, caminho in ARQUIVOS.items():
    pf = pq.ParquetFile(caminho)
    meta = pf.metadata
    linhas_meta.append({
        "base": base,
        "linhas": meta.num_rows,
        "row_groups": meta.num_row_groups,
        "colunas_fisicas": meta.num_columns,
        "tamanho_arquivo_mb": caminho.stat().st_size / 1024**2,
        "tamanho_total_descomprimido_mb": sum(
            meta.row_group(i).total_byte_size for i in range(meta.num_row_groups)
        ) / 1024**2,
    })
    print(f"\n{base} — schema:")
    print(pf.schema_arrow)

display(pd.DataFrame(linhas_meta))


grupo — schema:
ano: int64
CO_MUNICIPIO_TERRITORIO: large_string
NO_MUNICIPIO_TERRITORIO: large_string
TIPO_REFERENCIA_TERRITORIAL: large_string
tipo_grupo: large_string
grupo: large_string
disciplina: large_string
n_validos: int64
n_zeros: int64
media: double
media_positivas: double
mediana: double
std: double
q1: double
q3: double
base_origem: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1989

dsp — schema:
ano: int64
CO_MUNICIPIO_TERRITORIO: large_string
NO_MUNICIPIO_TERRITORIO: large_string
TIPO_REFERENCIA_TERRITORIAL: large_string
sexo: large_string
faixa_etaria: large_string
disciplina: large_string
n_participantes: int64
n_validos: int64
n_zeros: int64
media: double
media_positivas: double
mediana: double
std: double
q1: double
q3: double
base_origem: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2118


,base,linhas,row_groups,colunas_fisicas,tamanho_arquivo_mb,tamanho_total_descomprimido_mb
0,grupo,337214,1,16,8.573952,10.338845
1,dsp,451205,1,17,10.102821,12.507343


## 3. Worker de medição

Cada cenário inicia um novo processo. As medições de memória do PyArrow são feitas no mesmo instante que RSS/USS.

- `pool_bytes`: bytes atualmente alocados no pool padrão.
- `pool_max`: pico de alocação registrado pelo pool desde o início ou desde o reset.
- `total_allocated`: contador de bytes atualmente alocados reportado pelo PyArrow.
- `table_buffers`: tamanho dos buffers referenciados pela tabela resultante.

A memória do pool padrão não necessariamente contabiliza toda alocação nativa do processo. Por isso, a comparação com USS é central.

In [3]:
WORKER = r'''
import sys, json, gc, time, os
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

p = json.loads(sys.argv[1])
proc = psutil.Process()
pool = pa.default_memory_pool()

def snapshot(label, out):
    m = proc.memory_full_info()
    out[label] = {
        "rss_mb": m.rss / 1024**2,
        "uss_mb": getattr(m, "uss", 0) / 1024**2,
        "pool_bytes_mb": pool.bytes_allocated() / 1024**2,
        "pool_max_mb": pool.max_memory() / 1024**2,
        "total_allocated_mb": pa.total_allocated_bytes() / 1024**2,
    }

out = {
    "base": p["base"],
    "modo": p["modo"],
    "memory_map": p["memory_map"],
    "referencia": p["referencia"],
    "pool_backend": pool.backend_name,
}
snapshot("baseline", out)

# Abre o ParquetFile com memory_map controlado.
t0 = time.perf_counter()
pf = pq.ParquetFile(p["arquivo"], memory_map=p["memory_map"])
t1 = time.perf_counter()
out["tempo_abertura_s"] = t1 - t0
snapshot("apos_abertura", out)

# Lê as colunas projetadas do único row group.
t2 = time.perf_counter()
table = pf.read_row_group(0, columns=p["columns"])
t3 = time.perf_counter()
out["tempo_leitura_s"] = t3 - t2
out["linhas_antes_filtro"] = table.num_rows
out["colunas"] = table.num_columns
out["table_nbytes_mb"] = table.nbytes / 1024**2
out["table_buffers_mb"] = table.get_total_buffer_size() / 1024**2
snapshot("apos_leitura_arrow", out)

# Aplica o filtro em Arrow, como no teste anterior.
if p["referencia"] == "ESC":
    vals = ["ESC", "RESIDENCIA"]
elif p["referencia"] == "PROVA":
    vals = ["PROVA"]
else:
    vals = None

if vals is not None:
    arr = table["TIPO_REFERENCIA_TERRITORIAL"]
    mask = pc.is_in(arr, value_set=pa.array(vals, type=arr.type))
    table_filtrada = table.filter(mask)
    out["linhas_apos_filtro"] = table_filtrada.num_rows
    out["filtro_table_nbytes_mb"] = table_filtrada.nbytes / 1024**2
    out["filtro_table_buffers_mb"] = table_filtrada.get_total_buffer_size() / 1024**2
    snapshot("apos_filtro_arrow", out)
else:
    table_filtrada = table
    out["linhas_apos_filtro"] = table.num_rows

# Opcionalmente materializa uma cópia compactada dos buffers do resultado filtrado.
if p["compactar"]:
    t4 = time.perf_counter()
    table_compacta = table_filtrada.combine_chunks()
    t5 = time.perf_counter()
    out["tempo_combine_chunks_s"] = t5 - t4
    out["compacta_nbytes_mb"] = table_compacta.nbytes / 1024**2
    out["compacta_buffers_mb"] = table_compacta.get_total_buffer_size() / 1024**2
    snapshot("apos_combine_chunks", out)
    del table_compacta

# Limpa referências em etapas para observar a devolução ao pool.
del table_filtrada
if "table" in locals():
    del table
gc.collect()
snapshot("apos_del_tabelas", out)

del pf
gc.collect()
snapshot("apos_fechar_parquet", out)

print(json.dumps(out))
'''

def executar_worker(base, memory_map=False, referencia=REFERENCIA, compactar=False):
    payload = {
        "base": base,
        "arquivo": str(ARQUIVOS[base]),
        "columns": COLUNAS[base],
        "memory_map": memory_map,
        "referencia": referencia,
        "compactar": compactar,
        "modo": "read_row_group",
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER, json.dumps(payload)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        raise RuntimeError(f"Falha em {base}, memory_map={memory_map}:\\n{proc.stdout}\\n{proc.stderr}")
    linhas = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    if not linhas:
        raise RuntimeError(f"Worker sem JSON: {proc.stdout}")
    return json.loads(linhas[-1])

## 4. Teste base: memory_map desligado e ligado

Compara os dois modos em processos independentes. Execute primeiro com `REFERENCIA = "ESC"`. Depois, se necessário, altere para `"PROVA"` e repita.

In [4]:
resultados = []
for base in ["grupo", "dsp"]:
    for mmap in [False, True]:
        r = executar_worker(base, memory_map=mmap)
        resultados.append(r)
        print(
            f"{base:6s} | memory_map={mmap} | "
            f"USS baseline={r['baseline']['uss_mb']:.1f} MB | "
            f"USS Arrow={r['apos_leitura_arrow']['uss_mb']:.1f} MB | "
            f"pool={r['apos_leitura_arrow']['pool_bytes_mb']:.1f} MB | "
            f"buffers={r['table_buffers_mb']:.1f} MB"
        )

def resumir_worker(r):
    b = r["baseline"]
    a = r["apos_abertura"]
    x = r["apos_leitura_arrow"]
    f = r.get("apos_filtro_arrow", x)
    d = r["apos_del_tabelas"]
    return {
        "base": r["base"],
        "memory_map": r["memory_map"],
        "linhas": r["linhas_apos_filtro"],
        "tempo_abertura_s": r["tempo_abertura_s"],
        "tempo_leitura_s": r["tempo_leitura_s"],
        "delta_uss_abertura_mb": a["uss_mb"] - b["uss_mb"],
        "delta_uss_leitura_mb": x["uss_mb"] - a["uss_mb"],
        "delta_pool_leitura_mb": x["pool_bytes_mb"] - a["pool_bytes_mb"],
        "pool_max_apos_leitura_mb": x["pool_max_mb"],
        "table_nbytes_mb": r["table_nbytes_mb"],
        "table_buffers_mb": r["table_buffers_mb"],
        "delta_uss_filtro_mb": f["uss_mb"] - x["uss_mb"],
        "uss_apos_del_mb": d["uss_mb"],
        "pool_apos_del_mb": d["pool_bytes_mb"],
    }

df_resumo = pd.DataFrame([resumir_worker(r) for r in resultados])
display(df_resumo)

grupo  | memory_map=False | USS baseline=27.1 MB | USS Arrow=157.9 MB | pool=46.7 MB | buffers=40.1 MB
grupo  | memory_map=True | USS baseline=27.1 MB | USS Arrow=158.7 MB | pool=40.1 MB | buffers=40.1 MB
dsp    | memory_map=False | USS baseline=27.1 MB | USS Arrow=147.9 MB | pool=50.9 MB | buffers=45.1 MB
dsp    | memory_map=True | USS baseline=27.1 MB | USS Arrow=149.9 MB | pool=45.1 MB | buffers=45.1 MB


,base,memory_map,linhas,tempo_abertura_s,tempo_leitura_s,delta_uss_abertura_mb,delta_uss_leitura_mb,delta_pool_leitura_mb,pool_max_apos_leitura_mb,table_nbytes_mb,table_buffers_mb,delta_uss_filtro_mb,uss_apos_del_mb,pool_apos_del_mb
0,grupo,False,205400,0.000670,0.029176,4.042969,126.777344,46.725159,60.220581,40.133961,40.133999,49.601562,207.515625,10.473694
1,grupo,True,205400,0.000359,0.033478,0.046875,131.515625,40.134460,55.892151,40.133961,40.133999,49.546875,208.214844,3.882996
2,dsp,False,263761,0.000618,0.033039,4.046875,116.726562,50.904419,64.840149,45.132294,45.132332,51.554688,199.425781,10.970032
3,dsp,True,263761,0.000295,0.033223,0.082031,122.734375,45.132996,54.391602,45.132294,45.132332,55.523438,205.460938,5.198608


## 5. Separar leitura física, filtro e compactação dos chunks

O teste compara a memória antes e depois do filtro Arrow e de `combine_chunks()`. O filtro pode criar novos buffers; `combine_chunks()` pode gerar uma cópia contígua. Esta etapa verifica se parte do pico vem dessas operações posteriores à leitura.

In [5]:
resultados_compactacao=[]
for base in ["grupo", "dsp"]:
    r = executar_worker(base, memory_map=False, compactar=True)
    resultados_compactacao.append(r)

linhas=[]
for r in resultados_compactacao:
    x = r["apos_leitura_arrow"]
    f = r.get("apos_filtro_arrow", x)
    c = r["apos_combine_chunks"]
    linhas.append({
        "base": r["base"],
        "buffers_apos_leitura_mb": r["table_buffers_mb"],
        "buffers_apos_filtro_mb": r.get("filtro_table_buffers_mb", r["table_buffers_mb"]),
        "buffers_apos_combine_mb": r["compacta_buffers_mb"],
        "delta_uss_filtro_mb": f["uss_mb"] - x["uss_mb"],
        "delta_uss_combine_mb": c["uss_mb"] - f["uss_mb"],
        "delta_pool_combine_mb": c["pool_bytes_mb"] - f["pool_bytes_mb"],
        "pool_max_combine_mb": c["pool_max_mb"],
    })
display(pd.DataFrame(linhas))

,base,buffers_apos_leitura_mb,buffers_apos_filtro_mb,buffers_apos_combine_mb,delta_uss_filtro_mb,delta_uss_combine_mb,delta_pool_combine_mb,pool_max_combine_mb
0,grupo,40.133999,24.269329,24.269329,49.566406,0.0,0.0,71.859192
1,dsp,45.132332,26.249600,26.249600,55.574219,0.0,0.0,78.268860


## 6. Controle: abrir metadados sem ler colunas

Este controle mede o custo de abrir o arquivo e acessar metadados sem materializar o row group. Se USS crescer muito já nesta etapa, o custo não está restrito à leitura dos valores.

In [6]:
WORKER_META = r'''
import sys, json, gc, time
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
p=json.loads(sys.argv[1])
proc=psutil.Process()
pool=pa.default_memory_pool()
def snap():
    m=proc.memory_full_info()
    return {
      "rss":m.rss/1024**2,
      "uss":getattr(m,"uss",0)/1024**2,
      "pool":pool.bytes_allocated()/1024**2,
      "pool_max":pool.max_memory()/1024**2
    }
b=snap()
t0=time.perf_counter()
pf=pq.ParquetFile(p["arquivo"],memory_map=p["memory_map"])
meta=pf.metadata
nrows=meta.num_rows
nrg=meta.num_row_groups
schema=str(pf.schema_arrow)
t1=time.perf_counter()
a=snap()
del pf,meta,schema
gc.collect()
d=snap()
print(json.dumps({
 "base":p["base"],"memory_map":p["memory_map"],
 "linhas":nrows,"row_groups":nrg,"tempo_s":t1-t0,
 "baseline":b,"apos_metadados":a,"apos_del":d
}))
'''

def executar_meta(base, memory_map=False):
    payload={"base":base,"arquivo":str(ARQUIVOS[base]),"memory_map":memory_map}
    proc=subprocess.run([sys.executable,"-c",WORKER_META,json.dumps(payload)],
                        cwd=str(PROJECT_ROOT),capture_output=True,text=True)
    if proc.returncode:
        raise RuntimeError(proc.stderr)
    return json.loads([x for x in proc.stdout.splitlines() if x.strip().startswith("{")][-1])

metas=[executar_meta(b,m) for b in ["grupo","dsp"] for m in [False,True]]
display(pd.DataFrame([{
    "base":r["base"],"memory_map":r["memory_map"],"tempo_s":r["tempo_s"],
    "delta_uss_metadados_mb":r["apos_metadados"]["uss"]-r["baseline"]["uss"],
    "delta_pool_metadados_mb":r["apos_metadados"]["pool"]-r["baseline"]["pool"],
    "uss_apos_del_mb":r["apos_del"]["uss"],
} for r in metas]))

,base,memory_map,tempo_s,delta_uss_metadados_mb,delta_pool_metadados_mb,uss_apos_del_mb
0,grupo,False,0.001008,4.042969,0.0,27.570312
1,grupo,True,0.000605,0.046875,0.0,23.632812
2,dsp,False,0.000963,4.046875,0.0,27.585938
3,dsp,True,0.000691,0.078125,0.0,23.582031


## 7. Repetibilidade: duas execuções independentes

Repete a leitura Arrow em processos novos para verificar se o aumento de USS é consistente. Grandes diferenças entre execuções podem indicar influência de cache do sistema operacional, estado do ambiente ou variação do alocador.

In [7]:
repeticoes=[]
for base in ["grupo","dsp"]:
    for repeticao in [1,2]:
        r=executar_worker(base,memory_map=False)
        repeticoes.append({
            "base":base,
            "repeticao":repeticao,
            "delta_uss_leitura_mb":(
                r["apos_leitura_arrow"]["uss_mb"]-
                r["apos_abertura"]["uss_mb"]
            ),
            "delta_pool_leitura_mb":(
                r["apos_leitura_arrow"]["pool_bytes_mb"]-
                r["apos_abertura"]["pool_bytes_mb"]
            ),
            "pool_max_mb":r["apos_leitura_arrow"]["pool_max_mb"],
            "buffers_mb":r["table_buffers_mb"],
            "uss_apos_del_mb":r["apos_del_tabelas"]["uss_mb"],
        })
display(pd.DataFrame(repeticoes))

,base,repeticao,delta_uss_leitura_mb,delta_pool_leitura_mb,pool_max_mb,buffers_mb,uss_apos_del_mb
0,grupo,1,126.773438,46.725159,61.581116,40.133999,207.515625
1,grupo,2,126.769531,46.725159,56.270996,40.133999,207.500000
2,dsp,1,118.726562,50.904419,61.340332,45.132332,201.437500
3,dsp,2,122.734375,50.904419,59.357300,45.132332,209.488281


## 8. Diagnóstico orientado pelos resultados

Use as diferenças entre USS e Memory Pool para interpretar o comportamento:

- **Pool cresce próximo do USS:** boa parte das alocações passa pelo Memory Pool padrão do PyArrow.
- **USS cresce muito mais que o pool:** investigar alocações fora do pool padrão, bibliotecas nativas, buffers de leitura/mapeamento ou alocações temporárias do leitor.
- **Pool cresce durante leitura e cai após `del`:** buffers Arrow foram liberados para o pool; o RSS/USS pode não cair na mesma proporção porque o alocador mantém memória reservada.
- **`memory_map=True` reduz RSS/USS:** o modo de acesso ao arquivo influencia a materialização/cópia; validar em execução real antes de adotar.
- **`memory_map=True` aumenta RSS, mas não USS:** páginas mapeadas do arquivo podem aparecer na memória residente sem serem privadas/exclusivas.
- **Pico do pool alto, mas bytes atuais baixos:** houve alocação temporária durante a leitura.
- **`combine_chunks()` aumenta USS/pool:** há custo adicional de cópia/compactação após leitura.

### Limites do teste

1. `pa.total_allocated_bytes()` e `default_memory_pool()` observam o pool padrão; não necessariamente todas as alocações nativas.
2. RSS e USS são fotografias do processo e incluem bibliotecas e runtime.
3. `memory_map` não garante redução de memória; é uma variável experimental.
4. O teste de filtro Arrow é aplicado depois de ler o row group, pois o objetivo aqui é observar a etapa de materialização Arrow separadamente.
5. Não some buffers Arrow, memória profunda pandas e USS: são medidas diferentes e parcialmente sobrepostas.

### Critério para encerrar esta etapa

Precisamos identificar se o aumento de aproximadamente 150 MB por base está:
- contabilizado pelo Memory Pool;
- fora do Memory Pool;
- associado ao mapeamento do arquivo;
- ou concentrado em alocações temporárias que desaparecem após a leitura.

**Não altere `read_allfiles2()` ainda.** Use os resultados para decidir o próximo teste.